Индивидуальное задание

Вариант: 2

Группа: ИНБб-4301-02-00

ФИО: Исупов Андрей, Растегаева Дарья

In [1]:
# --- Google Colab, вариант А: архив лежит в корне Google Диска ---
from google.colab import drive
drive.mount('/content/drive')
!apt-get install -y p7zip-full > /dev/null
!7z x -p1 -o/content/ "/content/drive/MyDrive/datasets.zip" -y

# --- Google Colab, вариант Б: загрузка архива в текущую сессию ---
# from google.colab import files
# uploaded = files.upload()          # выберите datasets.zip
# !apt-get install -y p7zip-full > /dev/null
# !7z x -p1 datasets.zip -y

Mounted at /content/drive

7-Zip 23.01 (x64) : Copyright (c) 1999-2023 Igor Pavlov : 2023-06-20
 64-bit locale=en_US.UTF-8 Threads:2 OPEN_MAX:1048576

Scanning the drive for archives:
  0M Scan /content/drive/MyDrive/                                 1 file, 39644509 bytes (38 MiB)

Extracting archive: /content/drive/MyDrive/datasets.zip
 40% 4096 Open              --
Path = /content/drive/MyDrive/datasets.zip
Type = zip
Physical Size = 39644509

  0%     10% 1029 - datasets/trec07delay/data/inmail.1921                                                  20% 2009          30% 2978          40% 3943          50% 4941          60% 6011          71% 7201          80% 8082         

In [2]:
import os
import email
from email import policy
import pandas as pd

DATA_DIR = 'datasets/trec07delay/data/'
LABELS_FILE = 'datasets/trec07delay/delay/index'

assert os.path.isdir(DATA_DIR) and os.path.isfile(LABELS_FILE), \
    'Датасет не найден — распакуйте datasets.zip рядом с блокнотом (шаг 0)'

def text_email(path):
    """Извлекает тему и текстовое (text/plain) тело письма — та же функция, что в части 1."""
    try:
        with open(path, 'rb') as f:
            msg = email.message_from_binary_file(f, policy=policy.default)
        subject = str(msg.get('Subject') or '')
        parts = []
        for part in msg.walk():
            if part.get_content_type() == 'text/plain':
                try:
                    content = part.get_content()
                    if isinstance(content, str):
                        parts.append(content)
                except Exception:
                    pass
        return subject + ' ' + ' '.join(parts)
    except Exception:
        return ''

# Читаем метки: имя файла → 'spam'/'ham'
labels = {}
with open(LABELS_FILE) as f:
    for line in f:
        parts = line.split()
        if len(parts) == 2:
            labels[parts[1].split('/')[-1]] = parts[0]

print('Меток прочитано:', len(labels))

Меток прочитано: 10000


In [3]:
# Читаем все письма в список текстов X и список меток y (займёт до минуты)
X, y = [], []
empty = 0
filelist = sorted(labels, key=lambda name: int(name.split('.')[1]))
for filename in filelist:
    text = text_email(DATA_DIR + filename)
    if len(text.split()) <= 2:      # отсев пустых и HTML-only писем по сырому тексту;
        empty += 1                  # в части 1 он строже (после удаления стоп-слов), поэтому числа отличаются
        continue
    X.append(text)
    y.append(labels[filename])

print(f'Прочитано писем: {len(X)}, пропущено пустых: {empty}')
print('Пример текста (первые 200 символов):')
print(X[0][:200])

Прочитано писем: 9868, пропущено пустых: 132
Пример текста (первые 200 символов):
Generic Cialis, branded quality@  


## Шаг 1. Мешок слов: превращаем текст в числа

Модели машинного обучения работают с числами, а не с текстом. Самое простое числовое представление текста — **«мешок слов» (bag-of-words)**:

1. Проходим по всем документам и собираем **словарь** — список всех встретившихся слов.
2. Каждый документ представляем вектором длиной со словарь: на i-м месте стоит **число вхождений** i-го слова в этот документ.

Порядок слов и грамматика при этом теряются — остаётся только «какие слова и сколько раз встретились». Для классификации спама этого, как ни странно, почти достаточно. Разберём на простом примере из трёх «писем»:

In [4]:
from sklearn.feature_extraction.text import CountVectorizer

toy = ['hello mr bear',
       'hello hello gunter',
       'goodbye mr gunter']

toy_vectorizer = CountVectorizer(ngram_range=(1, 2))
toy_counts = toy_vectorizer.fit_transform(toy)

print('Словарь:', list(toy_vectorizer.get_feature_names_out()))
print(toy_counts.toarray())   # строки — письма, столбцы — слова словаря

Словарь: ['bear', 'goodbye', 'goodbye mr', 'gunter', 'hello', 'hello gunter', 'hello hello', 'hello mr', 'mr', 'mr bear', 'mr gunter']
[[1 0 0 0 1 0 0 1 1 1 0]
 [0 0 0 1 2 1 1 0 0 0 0]
 [0 1 1 1 0 0 0 0 1 0 1]]


Столбцы идут в алфавитном порядке словаря (`bear`, `goodbye`, `gunter`, `hello`, `mr`), поэтому первая строка `[1 0 0 1 1]` читается так: в письме A по одному разу встречаются `bear`, `hello` и `mr`, а `goodbye` и `gunter` — ни разу.

В реальном корпусе словарь содержит сотни тысяч слов, а в каждом отдельном письме встречается лишь малая их часть — матрица почти целиком нулевая. Поэтому `CountVectorizer` хранит её в **разреженном (sparse)** формате: в памяти лежат только ненулевые элементы.

In [5]:
# Случайное разбиение 70/30; stratify=y сохраняет долю спама одинаковой в обеих частях
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X, y, train_size=0.7, random_state=2, stratify=y)

print(f'train: {len(X_train)} писем, test: {len(X_test)} писем')
print('Доля спама: train {:.1%}, test {:.1%}'.format(
    sum(v == 'spam' for v in y_train) / len(y_train),
    sum(v == 'spam' for v in y_test) / len(y_test)))

train: 6907 писем, test: 2961 писем
Доля спама: train 80.3%, test 80.3%


In [6]:
# Строим словарь из униграмм и биграмм по тренировочной выборке
vectorizer = CountVectorizer(ngram_range=(1, 2))
X_train_counts = vectorizer.fit_transform(X_train)
X_test_counts = vectorizer.transform(X_test)

print('Размер словаря:', len(vectorizer.vocabulary_))
print('Матрица train:', X_train_counts.shape, '| ненулевых элементов:', X_train_counts.nnz)
print('Матрица test: ', X_test_counts.shape)

Размер словаря: 305934
Матрица train: (6907, 305934) | ненулевых элементов: 1145780
Матрица test:  (2961, 305934)


> **Важно:** `fit_transform` вызывается только на train — словарь строится по тренировочным данным. Тестовая выборка проходит только через `transform`: слова, которых не было в train, просто игнорируются. Иначе информация о тесте «протечёт» в обучение.

## Шаг 2. Наивный байесовский классификатор

Идея: для каждого класса (спам / ham) по тренировочной выборке оценивается, с какой вероятностью в нём встречается каждое слово словаря. Для нового письма по **формуле Байеса** вычисляется, к какому классу оно вероятнее относится, — по совокупности его слов.

Классификатор называется «наивным», потому что слова считаются независимыми друг от друга (в реальном языке это, конечно, не так). На практике для текстов метод работает на удивление хорошо, а обучается за доли секунды.

`MultinomialNB` — вариант наивного Байеса для счётчиков слов (наш случай). Чтобы слово, не встречавшееся в train, не обнулило всё произведение вероятностей, применяется **сглаживание Лапласа** (параметр `alpha`, по умолчанию 1.0). Подробная теория — в Приложении А методички.

In [7]:
from sklearn.naive_bayes import MultinomialNB

model = MultinomialNB()
model.fit(X_train_counts, y_train)     # обучение: подсчёт вероятностей слов в каждом классе
y_pred = model.predict(X_test_counts)  # прогноз для тестовой выборки

print('Первые 10 прогнозов:      ', [str(v) for v in y_pred[:10]])
print('Первые 10 истинных меток: ', y_test[:10])

Первые 10 прогнозов:       ['spam', 'ham', 'spam', 'spam', 'spam', 'ham', 'ham', 'ham', 'spam', 'spam']
Первые 10 истинных меток:  ['spam', 'ham', 'spam', 'spam', 'spam', 'ham', 'ham', 'ham', 'spam', 'spam']


In [8]:
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

print(f'Accuracy: {accuracy_score(y_test, y_pred):.1%}\n')
print(classification_report(y_test, y_pred, digits=3))

cm = pd.DataFrame(
    confusion_matrix(y_test, y_pred, labels=['spam', 'ham']),
    index=['на самом деле спам', 'на самом деле ham'],
    columns=['прогноз: спам', 'прогноз: ham'])
cm

Accuracy: 98.0%

              precision    recall  f1-score   support

         ham      0.927     0.978     0.952       584
        spam      0.994     0.981     0.988      2377

    accuracy                          0.980      2961
   macro avg      0.961     0.979     0.970      2961
weighted avg      0.981     0.980     0.981      2961



,прогноз: спам,прогноз: ham
на самом деле спам,2332,45
на самом деле ham,13,571


Напоминание по столбцам отчёта `classification_report`:

- **precision** — какая доля писем, названных классом, действительно принадлежит ему;
- **recall** — какая доля писем класса была найдена;
- **f1-score** — гармоническое среднее precision и recall;
- **support** — сколько писем этого класса было в тесте.

## Шаг 3. Сравниваем два подхода из части 1 и части 2 и доработанный бишграммами метод из части 2

| Подход | Accuracy | Precision (спам) | Recall (спам) | Чем примечателен |
|---|---|---|---|---|
| Чёрный список (часть 1) | 50.3 % | 85.9 % | 49.1 % | Прозрачные правила, но «мёртвый» список слов |
| Наивный Байес (часть 2) | 97.9 % | 99.4 % | 98.0 % | Обучается на данных сам, использует все слова |
| Наивный Байес с биграммами | 98.0 % | 99.4 % | 98.1 % | Обучается на данных сам, использует все слова и пары соседних |


**Вывод**: в ходе выполнения индивидуального задания был изменён способ векторизации текстов: к отдельным словам были добавлены биграммы с помощью параметра ngram_range=(1, 2) в CountVectorizer. В результате размер словаря увеличился с 45 547 до 305 934 признаков, то есть примерно в 6,72 раза.

Качество классификации немного улучшилось. Значение Accuracy увеличилось с 97.9% до 98.0%, Recall для класса спама — с 98.0% до 98.1%, а Precision осталось на уровне 99.4%. Это показывает, что использование биграмм позволило модели учитывать сочетания соседних слов и, хоть и незначительно, но всё равно повысить качество обнаружения спама. Модель с биграммами правильно распознала 2332 из 2377 спам-писем и 571 из 584 обычных писем. Она пропустила 45 спам-писем и ошибочно отнесла 13 обычных писем к спаму.

При этом увеличение словаря существенно расширяет количество признаков, которые обрабатывает модель. Следовательно, биграммы могут быть полезны для классификации текстов, поскольку некоторые сочетания слов информативнее отдельных слов, однако в данном эксперименте улучшение метрик оказалось небольшим.

Пример биграммы, которая будет информативнее отдельного слова: "instantly http". Можно предположить, что данная биграмма - это часть призыва пользователя немедленно перейти на сайт с незащищённым подключением, при этом отдельно слова http или instantly ничем не примечательны и спокойно могут быть не только частью спам-писем, но и зачастую частью обычных писем с рабочей коммуникацией.